# PhoBERT – Phân loại 3 nhãn: clean / offensive / hate (tiếng Việt)

Mục tiêu: fine-tune nhanh `vinai/phobert-base-v2` trên dataset đã gán nhãn theo guideline để **đánh giá xem hướng tiếp cận có khả quan không**.

| Label | Tên | Ý nghĩa tóm tắt |
|---|---|---|
| 0 | clean | Sạch; chấp nhận từ tục viết tắt (vl, vcl, đm, cc…), `vãi` đơn lẻ, `đéo` phủ định |
| 1 | offensive | Từ tục trần trụi; xúc phạm cá nhân; công kích nhóm nhỏ theo hành vi; quấy rối/gạ gẫm |
| 2 | hate | Kích động/đe dọa bạo lực; miệt thị nhóm theo vùng miền, chủng tộc, tôn giáo, giới tính… |

**Điểm khó của guideline này với model:**
- Ranh giới 0 vs 1 mang tính *từ vựng* (`như cc` = 0, `như cứt` = 1; `vãi` = 0, `vãi lồn` = 1). Vì vậy notebook **không lowercase, không bỏ dấu, không chuẩn hóa teencode** để giữ nguyên các dạng viết tắt.
- Ranh giới 1 vs 2 mang tính *ngữ nghĩa* (nhóm theo hành vi vs nhóm theo đặc tính). Lớp `hate` thường ít mẫu nên dùng **macro-F1** và class weight.
- Nhãn chỉ dựa vào CURRENT TEXT. Notebook mặc định chỉ đưa `text` vào model; có tùy chọn `USE_CONTEXT` để thử thêm `quoted_text`.

```markdown
# Thiết lập và cài đặt thư viện cần thiết
Chúng ta chỉ cần các thư viện cơ bản như `scikit-learn`, `pandas`, `numpy`, `underthesea` để xử lý tiếng Việt.
```

In [2]:
!pip install -q scikit-learn underthesea pandas numpy

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


```markdown
# Đọc và Tiền xử lý dữ liệu
Vì chỉ dùng TF-IDF nên chúng ta vẫn giữ bước chuẩn hóa dữ liệu và tách từ để đạt hiệu quả cao nhất.
```

In [5]:
import pandas as pd
import numpy as np
import re
import unicodedata
from underthesea import word_tokenize
from concurrent.futures import ThreadPoolExecutor
from google.colab import drive

# Kết nối Google Drive
try:
    drive.mount('/content/drive')
except Exception as e:
    print("Lưu ý: Không thể kết nối tự động với Drive. Đảm bảo bạn đang chạy trên Google Colab.")

# Khởi tạo các đường dẫn
DATA_PATH_TRAIN = "/content/drive/MyDrive/data/dataset/5_3_2/train.csv"
DATA_PATH_VAL   = "/content/drive/MyDrive/data/dataset/5_3_2/val.csv"
DATA_PATH_TEST  = "/content/drive/MyDrive/data/dataset/5_3_2/test.csv"

# Đọc dữ liệu
def load_any(path):
    if path.endswith(".csv"): return pd.read_csv(path)
    if path.endswith((".xlsx", ".xls")): return pd.read_excel(path)
    if path.endswith((".jsonl", ".json")):
        try: return pd.read_json(path, lines=True)
        except ValueError: return pd.read_json(path)
    raise ValueError("Định dạng file không hỗ trợ")

train_df = load_any(DATA_PATH_TRAIN)
val_df = load_any(DATA_PATH_VAL)
test_df = load_any(DATA_PATH_TEST)

# Trộn tập Train và Val lại với nhau để mô hình học máy truyền thống có thêm nhiều dữ liệu huấn luyện hơn
full_train_df = pd.concat([train_df, val_df], ignore_index=True)

# Tiền xử lý văn bản tiếng Việt
URL_RE    = re.compile(r"https?://\S+|www\.\S+")
SPACE_RE  = re.compile(r"\s+")
REPEAT_RE = re.compile(r"(.)\1{3,}")

def clean_text(t):
    t = "" if pd.isna(t) else str(t)
    t = unicodedata.normalize("NFC", t)
    t = URL_RE.sub(" ", t)
    t = REPEAT_RE.sub(r"\1\1\1", t)
    return SPACE_RE.sub(" ", t).strip()

def segment(t):
    try:
        return word_tokenize(t, format="text")
    except Exception:
        return t

def seg_list(texts):
    with ThreadPoolExecutor(max_workers=4) as ex:
        return list(ex.map(segment, texts))

def prepare(d):
    d = d.copy()
    d["text_clean"] = d["text"].apply(clean_text)
    d["text_seg"] = seg_list(d["text_clean"].tolist())
    return d

print("Đang tiền xử lý và tách từ...")
full_train_df = prepare(full_train_df)
test_df = prepare(test_df)
print("Hoàn thành tiền xử lý!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đang tiền xử lý và tách từ...
Hoàn thành tiền xử lý!


```markdown
# Huấn luyện TF-IDF + Logistic Regression và TF-IDF + Linear SVM
```

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline, make_union
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import joblib

ID2LABEL = {0: "clean", 1: "offensive", 2: "hate"}

# Định nghĩa bộ Vectorizer kết hợp Word N-grams và Char-wb N-grams
vectorizer = make_union(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
    TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True)
)

# 1. Model 1: Logistic Regression
model_lr = make_pipeline(
    vectorizer,
    LogisticRegression(max_iter=1000, class_weight="balanced", C=3.0, random_state=42)
)

# 2. Model 2: Linear SVM
model_svm = make_pipeline(
    vectorizer,
    LinearSVC(class_weight="balanced", C=1.0, max_iter=2000, random_state=42)
)

# Huấn luyện
print("--- Đang huấn luyện Logistic Regression ---")
model_lr.fit(full_train_df["text_seg"], full_train_df["label"])

print("--- Đang huấn luyện Linear SVM ---")
model_svm.fit(full_train_df["text_seg"], full_train_df["label"])

print("Đã hoàn thành huấn luyện cả 2 mô hình!")

--- Đang huấn luyện Logistic Regression ---
--- Đang huấn luyện Linear SVM ---
Đã hoàn thành huấn luyện cả 2 mô hình!


```markdown
# Đánh giá và so sánh kết quả trên tập Test
```

In [7]:
# Dự đoán
pred_lr = model_lr.predict(test_df["text_seg"])
pred_svm = model_svm.predict(test_df["text_seg"])

y_true = test_df["label"]
target_names = list(ID2LABEL.values())

print("================ LOGISTIC REGRESSION ================")
print("Macro-F1:", round(f1_score(y_true, pred_lr, average="macro"), 4))
print(classification_report(y_true, pred_lr, target_names=target_names, digits=4, zero_division=0))

print("================== LINEAR SVM ==================")
print("Macro-F1:", round(f1_score(y_true, pred_svm, average="macro"), 4))
print(classification_report(y_true, pred_svm, target_names=target_names, digits=4, zero_division=0))

================ LOGISTIC REGRESSION ================
Macro-F1: 0.9113
              precision    recall  f1-score   support

       clean     0.9335    0.9532    0.9432      3077
   offensive     0.8907    0.8994    0.8950      1232
        hate     0.9153    0.8771    0.8958      1847

    accuracy                         0.9196      6156
   macro avg     0.9131    0.9099    0.9113      6156
weighted avg     0.9194    0.9196    0.9193      6156

================== LINEAR SVM ==================
Macro-F1: 0.9208
              precision    recall  f1-score   support

       clean     0.9365    0.9675    0.9517      3077
   offensive     0.9134    0.8985    0.9059      1232
        hate     0.9258    0.8847    0.9048      1847

    accuracy                         0.9288      6156
   macro avg     0.9252    0.9169    0.9208      6156
weighted avg     0.9286    0.9288    0.9285      6156



```markdown
# Lưu trữ mô hình để sử dụng lại
```

In [8]:
# Lưu lại các mô hình tốt nhất
joblib.dump(model_lr, "/content/drive/MyDrive/data/result/model_lr.pkl")
joblib.dump(model_svm, "/content/drive/MyDrive/data/result/model_svm.pkl")
print("Đã lưu 'model_lr.pkl' và 'model_svm.pkl' thành công!")

Đã lưu 'model_lr.pkl' và 'model_svm.pkl' thành công!
